In [1]:
import pandas as pd
df = pd.read_csv('nlp_customer_messsage.csv')
print(df.head())

  message_id                                            message
0     MSG001  Good morning, My phone is not working :(  Than...
1     MSG002  Hi, Thanks for your quick response! I need hel...
2     MSG003  Hi support, I LOVE this product 😍😍 Can someone...
3     MSG004  Hey there, The new update is AMAZING!! Please ...
4     MSG005  Dear team, The price is 250 EGP, too expensive...


In [2]:
df.shape

(80, 2)

In [3]:
print(df.isnull().sum())

message_id    0
message       0
dtype: int64


In [4]:
print(df['message_id'].duplicated().sum())

0


In [5]:
df['char_count'] = df['message'].astype(str).apply(len)

longest_message = df.loc[df['char_count'].idxmax()]
shortest_message = df.loc[df['char_count'].idxmin()]

print(longest_message)
print(shortest_message)

message_id                                               MSG036
message       Hey there, I need help with payment for order ...
char_count                                                  209
Name: 35, dtype: object
message_id                                            MSG029
message       Dear team, Thanks! Great service :) Thank you.
char_count                                                46
Name: 28, dtype: object


In [6]:
import re

messages = df['message'].astype(str)

email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
df['emails'] = messages.apply(lambda x: re.findall(email_pattern, x))

url_pattern = r'https?://\S+|www\.\S+'
df['urls'] = messages.apply(lambda x: re.findall(url_pattern, x))

phone_pattern = r'\b010\d{8}\b'
df['phones'] = messages.apply(lambda x: re.findall(phone_pattern, x))

number_pattern = r'\d+'
df['numbers'] = messages.apply(lambda x: re.findall(number_pattern, x))

urgent_pattern = r'\burgent\b'
df['has_urgent'] = messages.apply(lambda x: bool(re.search(urgent_pattern, x, re.IGNORECASE)))

extracted_sample = df[(df['emails'].str.len() > 0) | 
                      (df['urls'].str.len() > 0) | 
                      (df['phones'].str.len() > 0) | 
                      (df['has_urgent'] == True)]

print("--- 5 Sample Messages with Extracted Info ---")
print(extracted_sample[['message', 'emails', 'urls', 'phones', 'numbers', 'has_urgent']].head(5))

--- 5 Sample Messages with Extracted Info ---
                                             message                   emails  \
2  Hi support, I LOVE this product 😍😍 Can someone...  [customer3@example.com]   
3  Hey there, The new update is AMAZING!! Please ...                       []   
4  Dear team, The price is 250 EGP, too expensive...                       []   
5  Dear support, The package for order 10102 is d...  [customer6@example.com]   
6  Hello, Bad quality... very disappointed. What ...                       []   

                                      urls         phones             numbers  \
2                                       []             []                 [3]   
3  [https://help.example.com/ticket/10068]             []             [10068]   
4                                       []  [01010011705]  [250, 01010011705]   
5                                       []             []          [10102, 6]   
6                                       []             []     

In [7]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
        
    text = re.sub(email_pattern, '[EMAIL]', text)
    
    text = re.sub(url_pattern, '[URL]', text)
    
    text = re.sub(phone_pattern, '[PHONE]', text)
    
    text = re.sub(number_pattern, '[NUMBER]', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    
    text = text.strip()
    
    return text

df['clean_message'] = df['message'].apply(clean_text)
print(df[['message', 'clean_message']].head())

                                             message  \
0  Good morning, My phone is not working :(  Than...   
1  Hi, Thanks for your quick response! I need hel...   
2  Hi support, I LOVE this product 😍😍 Can someone...   
3  Hey there, The new update is AMAZING!! Please ...   
4  Dear team, The price is 250 EGP, too expensive...   

                                       clean_message  
0  Good morning, My phone is not working :( Thank...  
1  Hi, Thanks for your quick response! I need hel...  
2  Hi support, I LOVE this product 😍😍 Can someone...  
3  Hey there, The new update is AMAZING!! Please ...  
4  Dear team, The price is [NUMBER] EGP, too expe...  


In [8]:
print("--- Sample of 8 rows ---")
print(df[['message', 'clean_message']].head(8))

--- Sample of 8 rows ---
                                             message  \
0  Good morning, My phone is not working :(  Than...   
1  Hi, Thanks for your quick response! I need hel...   
2  Hi support, I LOVE this product 😍😍 Can someone...   
3  Hey there, The new update is AMAZING!! Please ...   
4  Dear team, The price is 250 EGP, too expensive...   
5  Dear support, The package for order 10102 is d...   
6  Hello, Bad quality... very disappointed. What ...   
7  Hello team, Great quality, but delivery took 7...   

                                       clean_message  
0  Good morning, My phone is not working :( Thank...  
1  Hi, Thanks for your quick response! I need hel...  
2  Hi support, I LOVE this product 😍😍 Can someone...  
3  Hey there, The new update is AMAZING!! Please ...  
4  Dear team, The price is [NUMBER] EGP, too expe...  
5  Dear support, The package for order [NUMBER] i...  
6  Hello, Bad quality... very disappointed. What ...  
7  Hello team, Great quality, 

In [9]:
test_strings = [
    "Contact support at help@example.com for refunds.",
    "My new number is 01012345678, call me.",
    "Visit our new page https://www.website.com.",
    "This is a completely normal sentence."
]

for text in test_strings:
    print(f"Original: {text}")
    print(f"Cleaned:  {clean_text(text)}\n")

Original: Contact support at help@example.com for refunds.
Cleaned:  Contact support at [EMAIL] for refunds.

Original: My new number is 01012345678, call me.
Cleaned:  My new number is [PHONE], call me.

Original: Visit our new page https://www.website.com.
Cleaned:  Visit our new page [URL]

Original: This is a completely normal sentence.
Cleaned:  This is a completely normal sentence.



In [10]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt')

df['sentences_nltk'] = df['clean_message'].apply(sent_tokenize)

df['tokens_nltk'] = df['clean_message'].apply(word_tokenize)

df['sentence_count'] = df['sentences_nltk'].apply(len)
df['token_count'] = df['tokens_nltk'].apply(len)

top_5 = df.nlargest(5, 'token_count')[['clean_message', 'token_count']]
print(top_5)

                                        clean_message  token_count
65  Hi, The package for order [NUMBER] is damaged....           38
35  Hey there, I need help with payment for order ...           37
71  Hello team, I cannot log in to my account. Ple...           36
44  Dear team, The price is [NUMBER] EGP, too expe...           35
59  Hey there, My order [NUMBER] arrived late!!! P...           35


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HELAL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\HELAL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [12]:
%pip install spacy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 2.1 MB/s eta 0:00:06
     -- ------------------------------------- 0.8/12.8 MB 1.7 MB/s eta 0:00:08
     ---- ----------------------------------- 1.3/12.8 MB 1.9 MB/s eta 0:00:06
     ---- ----------------------------------- 1.6/12.8 MB 2.0 MB/s eta 0:00:06
     ----- ---------------------------------- 1.8/12.8 MB 1.9 MB/s eta 0:00:06
     ------ --------------------------------- 2.1/12.8 MB 1.8 MB/s eta 0:00:06
     -------- ------------------------------- 2.6/12.8 MB 1.8 MB/s eta 0:00:06
     --------- ------------------------------ 3.1/12.8 MB 1.8 MB/s eta 0:00:06
     ---------- ----------------------------- 3.4/12.8 MB 1.9 MB/s eta 0:00:06
     ------------- -------------------------- 4.2/12.8 MB 1.9 MB/s eta 0:00:05
     -------------- ------------------------- 4.7/12.8 MB 2.0 MB/s


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import spacy

nlp = spacy.load("en_core_web_sm")

first_10_messages = df['clean_message'].head(10).tolist()
spacy_docs = [nlp(text) for text in first_10_messages]


print("\n--- Different between Spacy and nltk ---")
for i in range(10):
    nltk_tokens = df['tokens_nltk'].iloc[i]
    spacy_tokens = [token.text for token in spacy_docs[i]]
    
    if nltk_tokens != spacy_tokens:
        print(f"NLTK Tokens: {nltk_tokens}")
        print(f"spaCy Tokens: {spacy_tokens}")
        break 

for token in spacy_docs[0]:
    print(f"Token: {token.text:15} | is_alpha: {token.is_alpha!s:5} | is_punct: {token.is_punct!s:5}")


--- Different between Spacy and nltk ---
NLTK Tokens: ['Good', 'morning', ',', 'My', 'phone', 'is', 'not', 'working', ':', '(', 'Thanks', 'again', '!']
spaCy Tokens: ['Good', 'morning', ',', 'My', 'phone', 'is', 'not', 'working', ':(', 'Thanks', 'again', '!']
Token: Good            | is_alpha: True  | is_punct: False
Token: morning         | is_alpha: True  | is_punct: False
Token: ,               | is_alpha: False | is_punct: True 
Token: My              | is_alpha: True  | is_punct: False
Token: phone           | is_alpha: True  | is_punct: False
Token: is              | is_alpha: True  | is_punct: False
Token: not             | is_alpha: True  | is_punct: False
Token: working         | is_alpha: True  | is_punct: False
Token: :(              | is_alpha: False | is_punct: True 
Token: Thanks          | is_alpha: True  | is_punct: False
Token: again           | is_alpha: True  | is_punct: False
Token: !               | is_alpha: False | is_punct: True 


In [15]:
import pandas as pd
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer(language='english')
lancaster = LancasterStemmer()

sample_words = ['running', 'easily', 'fairly', 'cats', 'troubles', 'generously', 'agreed', 'owned', 'meeting', 'sizing']


porter_stems = [porter.stem(w) for w in sample_words]
snowball_stems = [snowball.stem(w) for w in sample_words]
lancaster_stems = [lancaster.stem(w) for w in sample_words]

task8_df = pd.DataFrame({
    'Original Word': sample_words,
    'Porter Stem': porter_stems,
    'Snowball Stem': snowball_stems,
    'Lancaster Stem': lancaster_stems
})

display(task8_df)

,Original Word,Porter Stem,Snowball Stem,Lancaster Stem
0,running,run,run,run
1,easily,easili,easili,easy
2,fairly,fairli,fair,fair
3,cats,cat,cat,cat
4,troubles,troubl,troubl,troubl
5,generously,gener,generous,gen
6,agreed,agre,agre,agree
7,owned,own,own,own
8,meeting,meet,meet,meet
9,sizing,size,size,siz


In [16]:
from nltk.stem import WordNetLemmatizer
import spacy
import nltk


nltk.download('wordnet')


lemmatizer = WordNetLemmatizer()
nlp = spacy.load("en_core_web_sm")

lemmas_nltk = [lemmatizer.lemmatize(w) for w in sample_words]


doc = nlp(" ".join(sample_words))
lemmas_spacy = [token.lemma_ for token in doc]


task8_df['NLTK Lemma'] = lemmas_nltk
task8_df['spaCy Lemma'] = lemmas_spacy

display(task8_df[['Original Word', 'Porter Stem', 'NLTK Lemma', 'spaCy Lemma']])

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HELAL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,Original Word,Porter Stem,NLTK Lemma,spaCy Lemma
0,running,run,running,run
1,easily,easili,easily,easily
2,fairly,fairli,fairly,fairly
3,cats,cat,cat,cat
4,troubles,troubl,trouble,trouble
5,generously,gener,generously,generously
6,agreed,agre,agreed,agree
7,owned,own,owned,own
8,meeting,meet,meeting,meeting
9,sizing,size,sizing,size


In [ ]:
import nltk
from nltk.corpus import stopwords
import spacy
import pandas as pd

nltk.download('stopwords')
nlp = spacy.load("en_core_web_sm")


df['lower_tokens'] = df['tokens_nltk'].apply(lambda tokens: [t.lower() for t in tokens])

nltk_stop_words = set(stopwords.words('english'))


spacy_stop_words = nlp.Defaults.stop_words

df['tokens_no_stop_nltk'] = df['lower_tokens'].apply(
    lambda tokens: [t for t in tokens if t not in nltk_stop_words]
)
df['tokens_no_stop_spacy'] = df['lower_tokens'].apply(
    lambda tokens: [t for t in tokens if t not in spacy_stop_words]
)


for i in range(10):
    orig = len(df['lower_tokens'].iloc[i])
    n_nltk = len(df['tokens_no_stop_nltk'].iloc[i])
    n_spacy = len(df['tokens_no_stop_spacy'].iloc[i])
    print(f"Msg {i}: Original={orig} | NLTK={n_nltk} | spaCy={n_spacy}")



print(f" No. of words NLTK: {len(nltk_stop_words)} | No. of words spaCy: {len(spacy_stop_words)}")

keep_words = {'not', 'no', 'never'}
custom_stop_words = nltk_stop_words - keep_words

msg_idx = 0
orig_tokens = df['lower_tokens'].iloc[msg_idx]
nltk_default = df['tokens_no_stop_nltk'].iloc[msg_idx]
custom_policy = [t for t in orig_tokens if t not in custom_stop_words]

print(f"\n--- {msg_idx} ---")
print(f"Original : {orig_tokens}")
print(f"(NLTK): {nltk_default}")
print(f"(Challenge): {custom_policy}")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HELAL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Msg 0: Original=13 | NLTK=9 | spaCy=9
Msg 1: Original=15 | NLTK=12 | spaCy=12
Msg 2: Original=18 | NLTK=14 | spaCy=13
Msg 3: Original=17 | NLTK=14 | spaCy=13
Msg 4: Original=23 | NLTK=17 | spaCy=16
Msg 5: Original=27 | NLTK=21 | spaCy=20
Msg 6: Original=17 | NLTK=12 | spaCy=12
Msg 7: Original=22 | NLTK=21 | spaCy=20
Msg 8: Original=24 | NLTK=21 | spaCy=21
Msg 9: Original=22 | NLTK=16 | spaCy=15
 No. of words NLTK: 198 | No. of words spaCy: 326

--- 0 ---
Original : ['good', 'morning', ',', 'my', 'phone', 'is', 'not', 'working', ':', '(', 'thanks', 'again', '!']
(NLTK): ['good', 'morning', ',', 'phone', 'working', ':', '(', 'thanks', '!']
(Challenge): ['good', 'morning', ',', 'phone', 'not', 'working', ':', '(', 'thanks', '!']


In [18]:
import spacy
import pandas as pd
from collections import Counter


nlp = spacy.load("en_core_web_sm")

for i, text in enumerate(df['clean_message'].head(5)):
    print(f"\nMessage {i+1}: {text}")
    doc = nlp(text)
    for token in doc:
        print(f"{token.text:12} | POS: {token.pos_:5} | TAG: {token.tag_}")


all_pos_tags = []
all_verbs = set()
all_adjectives = set()

for text in df['clean_message']:
    doc = nlp(str(text))
    for token in doc:
        all_pos_tags.append(token.pos_)
        if token.pos_ == "VERB":
            all_verbs.add(token.text)
        elif token.pos_ == "ADJ":
            all_adjectives.add(token.text)

pos_counts = Counter(all_pos_tags)
pos_df = pd.DataFrame(pos_counts.items(), columns=['POS Tag', 'Count']).sort_values(by='Count', ascending=False)

display(pos_df)

print(f"5 Verbs: {list(all_verbs)[:5]}")
print(f"5 Adjectives: {list(all_adjectives)[:5]}")


Message 1: Good morning, My phone is not working :( Thanks again!
Good         | POS: ADJ   | TAG: JJ
morning      | POS: NOUN  | TAG: NN
,            | POS: PUNCT | TAG: ,
My           | POS: PRON  | TAG: PRP$
phone        | POS: NOUN  | TAG: NN
is           | POS: AUX   | TAG: VBZ
not          | POS: PART  | TAG: RB
working      | POS: VERB  | TAG: VBG
:(           | POS: PUNCT | TAG: NFP
Thanks       | POS: NOUN  | TAG: NNS
again        | POS: ADV   | TAG: RB
!            | POS: PUNCT | TAG: .

Message 2: Hi, Thanks for your quick response! I need help ASAP!!!
Hi           | POS: INTJ  | TAG: UH
,            | POS: PUNCT | TAG: ,
Thanks       | POS: NOUN  | TAG: NNS
for          | POS: ADP   | TAG: IN
your         | POS: PRON  | TAG: PRP$
quick        | POS: ADJ   | TAG: JJ
response     | POS: NOUN  | TAG: NN
!            | POS: PUNCT | TAG: .
I            | POS: PRON  | TAG: PRP
need         | POS: VERB  | TAG: VBP
help         | POS: NOUN  | TAG: NN
ASAP         | POS: ADV   | TA

,POS Tag,Count
2,PUNCT,416
1,NOUN,281
6,VERB,181
12,X,180
3,PRON,171
0,ADJ,93
8,INTJ,87
4,AUX,68
7,ADV,52
9,ADP,50


5 Verbs: ['check', 'LOVE', 'open', 'do', 'took']
5 Adjectives: ['backup', 'quick', 'Dear', 'AMAZING', 'disappointed']


In [19]:
for i, text in enumerate(df['message'].head(15)):
    doc = nlp(str(text))
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    if entities:
        print(f"Msg {i}: {entities}")

entities_data = []
for idx, row in df.iterrows():
    doc = nlp(str(row['message']))
    for ent in doc.ents:
        entities_data.append({
            'message_id': row['message_id'],
            'entity_text': ent.text,
            'entity_label': ent.label_
        })

entities_df = pd.DataFrame(entities_data)

display(entities_df.head())

if not entities_df.empty:
    display(entities_df['entity_label'].value_counts())
else:
    print("No entity found")

Msg 4: [('250', 'CARDINAL'), ('EGP', 'ORG'), ('01010011705', 'DATE')]
Msg 5: [('10102', 'DATE')]
Msg 6: [('@support', 'GPE')]
Msg 7: [('7 days', 'DATE')]
Msg 9: [('01010023410', 'DATE')]
Msg 10: [('A-45-2026', 'DATE')]
Msg 12: [('2', 'CARDINAL'), ('1', 'CARDINAL')]
Msg 13: [('#', 'CARDINAL'), ('@support', 'GPE')]
Msg 14: [('01010035115', 'DATE')]


,message_id,entity_text,entity_label
0,MSG005,250,CARDINAL
1,MSG005,EGP,ORG
2,MSG005,01010011705,DATE
3,MSG006,10102,DATE
4,MSG007,@support,GPE


entity_label
DATE        43
CARDINAL    19
GPE          9
ORG          6
PERSON       6
Name: count, dtype: int64